# Step 2 · Index the Volume into Vector Search

This notebook takes the Markdown pages that **Step 1** saved in the Volume and makes them searchable by AI agents:

1. **Chunk.** A **Lakeflow Spark Declarative Pipeline** splits every page into small, overlapping, heading-aware passages called chunks. It only processes new or changed files, so re-runs are fast.
2. **Index.** A **Vector Search** index is created on the chunk table (or refreshed if it already exists). Databricks computes the embeddings for you.
3. **Test.** A sample question is run against the index so you can see the results.

This step is independent of Step 1. You can run it any time the Volume has pages, and as often as you like.

## How to run
1. Attach the notebook to **Serverless** compute (environment version **6**).
2. Check the widgets. The **Catalog / Schema / Content volume** values must match Step 1.
3. Click **Run all**. The first run takes longer, because the Vector Search endpoint and index are being created.

## Settings (widgets)
| Widget | What it means |
|---|---|
| **Catalog / Schema / Content volume** | Where Step 1 saved the pages. The chunk tables are created in the same schema. |
| **Pipeline name** | Name of the pipeline that chunks the pages. It's created on the first run and reused after that. |
| **Chunk size / overlap (tokens)** | About how big each passage is, and how much neighbouring passages overlap. 512 / 64 is a good default. |
| **Vector Search endpoint** | The Vector Search compute that serves the index. It's created if missing. |
| **Index name** | Name of the index, created in the same schema. |
| **Embedding model** | The model that turns text into vectors. |
| **Full refresh** | `true` rebuilds all chunks from scratch. Use it after deleting pages or changing the chunk size. |
| **Test question** | A sample question to try against the finished index. |

The code lives in this project: `pipelines/chunk_documents.py` (the pipeline), `src/web_content_kit/chunking.py` (how pages are split) and `src/web_content_kit/indexing.py` (pipeline and Vector Search management).

## 1. Install this project
This installs the project's code, `web_content_kit`, together with the Databricks SDK that manages the pipeline and Vector Search.

In [ ]:
%uv pip install --quiet "..[index]"

In [ ]:
%restart_python

## 2. Settings

In [ ]:
dbutils.widgets.text("catalog", "main", "01 Catalog")
dbutils.widgets.text("schema", "web_content", "02 Schema")
dbutils.widgets.text("volume", "web_content", "03 Content volume")
dbutils.widgets.text("pipeline_name", "web_content_indexing", "04 Pipeline name")
dbutils.widgets.text("chunk_size_tokens", "512", "05 Chunk size (tokens)")
dbutils.widgets.text("chunk_overlap_tokens", "64", "06 Chunk overlap (tokens)")
dbutils.widgets.text("vs_endpoint", "web_content_vs", "07 Vector Search endpoint")
dbutils.widgets.text("index_name", "web_content_chunks_index", "08 Index name")
dbutils.widgets.text("embedding_model", "databricks-gte-large-en", "09 Embedding model")
dbutils.widgets.dropdown("full_refresh", "false", ["true", "false"], "10 Full refresh")
dbutils.widgets.text("test_query", "How do I create a vector search index?", "11 Test question")

In [ ]:
from web_content_kit import settings as s
from web_content_kit.indexing import IndexSettings
from web_content_kit.logs import get_logger

log = get_logger()
widget = dbutils.widgets.get

index_settings = IndexSettings(
    catalog=s.parse_identifier("Catalog", widget("catalog")),
    schema=s.parse_identifier("Schema", widget("schema")),
    volume=s.parse_identifier("Content volume", widget("volume")),
    pipeline_name=s.parse_identifier("Pipeline name", widget("pipeline_name")),
    vs_endpoint=s.parse_identifier("Vector Search endpoint", widget("vs_endpoint")),
    index_name=s.parse_identifier("Index name", widget("index_name")),
    embedding_model=widget("embedding_model").strip(),
    chunk_size_tokens=s.parse_int("Chunk size", widget("chunk_size_tokens"), minimum=64),
    chunk_overlap_tokens=s.parse_int("Chunk overlap", widget("chunk_overlap_tokens"), minimum=0),
    full_refresh=s.parse_bool(widget("full_refresh")),
)
if index_settings.chunk_overlap_tokens >= index_settings.chunk_size_tokens:
    raise ValueError("Chunk overlap must be smaller than the chunk size.")
test_query = widget("test_query").strip()
log.info("Source volume: %s", index_settings.content_path)
log.info("Chunk table:   %s", index_settings.chunk_table)
log.info("Index:         %s (endpoint %s)", index_settings.full_index_name, index_settings.vs_endpoint)

## 3. Check there are pages to index

In [ ]:
from pathlib import Path

content_dir = Path(index_settings.content_path)
if not content_dir.exists():
    raise FileNotFoundError(
        f"The volume {index_settings.content_path} does not exist. "
        "Run Step 1 first, or check the Catalog/Schema/Volume widgets."
    )
first_page = next(content_dir.rglob("*.md"), None)
if first_page is None:
    raise FileNotFoundError(f"No .md pages found in {index_settings.content_path}. Run Step 1 first.")
log.info("Found pages in the volume (for example %s).", first_page.relative_to(content_dir))

## 4. Create (or update) and run the chunking pipeline
The pipeline produces two tables in your schema:
* **`documents`**: one row per page file that was read.
* **`document_chunks`**: one row per chunk, always the latest version. This is the source of the Vector Search index.

In [ ]:
import os

from databricks.sdk import WorkspaceClient

from web_content_kit.indexing import create_or_update_pipeline, run_pipeline

w = WorkspaceClient()

# This notebook lives in <project>/notebooks/, so the project folder is one level up.
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root = os.path.dirname(os.path.dirname(notebook_path))
if not project_root.startswith("/Workspace"):
    project_root = "/Workspace" + project_root
pipeline_file = f"{project_root}/pipelines/chunk_documents.py"
if not os.path.exists(pipeline_file):
    raise FileNotFoundError(f"Pipeline file not found at {pipeline_file}. Keep the project folders together.")

pipeline_id = create_or_update_pipeline(w, index_settings, pipeline_file, project_root)
run_pipeline(w, pipeline_id, index_settings.full_refresh)

chunks = spark.table(index_settings.chunk_table)
chunk_count = chunks.count()
doc_count = chunks.select("doc_id").distinct().count()
log.info("Chunk table has %d chunks from %d pages.", chunk_count, doc_count)
if chunk_count == 0:
    raise RuntimeError("The pipeline produced no chunks. Check that the pages in the volume contain text.")

## 5. Create (or refresh) the Vector Search index
The index is a **Delta Sync** index. Databricks reads the `chunk_text` column, computes embeddings with the chosen model, and keeps the index in step with the chunk table every time this notebook runs.

In [ ]:
from web_content_kit.indexing import ensure_endpoint, ensure_index, wait_for_index

ensure_endpoint(w, index_settings.vs_endpoint)
ensure_index(w, index_settings)
wait_for_index(w, index_settings.full_index_name, expected_rows=chunk_count)

## 6. Try a question
Hybrid search (keywords + meaning) returns the most relevant chunks. An AI agent uses the index the same way.

In [ ]:
import pandas as pd

from web_content_kit.indexing import search

if test_query:
    hits = pd.DataFrame(search(w, index_settings.full_index_name, test_query))
    log.info("Top %d results for: %s", len(hits), test_query)
    display(hits)

displayHTML(f"""
<h3>Indexing complete</h3>
<ul>
  <li>Chunk table: <code>{index_settings.chunk_table}</code> ({chunk_count} chunks from {doc_count} pages)</li>
  <li>Vector Search index: <code>{index_settings.full_index_name}</code>
      on endpoint <code>{index_settings.vs_endpoint}</code></li>
</ul>
<p>Give an agent this index as a retrieval tool, or add it to an Agent Bricks Knowledge Assistant.</p>
""")